## Likelihood analysis

In the annotation notebook, we have seen that we can add the likelihood as an annotation to each text signal. We can call the same function also to create a likelihood evaluation for each utterance in a sequence and the interations overall. This approach was proposed by:

```Mehri, S., & Eskenazi, M. (2020). USR: An unsupervised and reference free evaluation metric for dialog generation. arXiv preprint arXiv:2005.00456.```

They finetuned a RoBERTa model (USR) specifically to mimic human evaluation, as a reference-free approach. Reference-free means that is does not need a human-created response to compare the system response. Instead it determines the likelihood of the system response given the preceding conversation. The likelihood is calculated by taking the average likelihood of all tokens of the system response according to the model. The USR model was finetuned with conversations from TopicalChat and PersonaChat:

```Karthik Gopalakrishnan, Behnam Hedayatnia, Qinlang Chen, Anna Gottardi, Sanjeev Kwatra, Anu Venkatesh, Raefer Gabriel, Dilek Hakkani-Tur, and Amazon Alexa AI. 2019. Topical-chat: Towards knowledge-grounded open-domain conversations. Proc. Interspeech 2019, pages 1891–1895.```
```Saizheng Zhang, Emily Dinan, Jack Urbanek, Arthur Szlam, Douwe Kiela, and Jason Weston. 2018. Personalizing dialogue agents: I have a dog, do you have pets too? arXiv preprint arXiv:1801.07243.```

Their approach has moderate correlation with human judgments: .42 (TopicalChat) and .48 (PersonaChat). The USR model can be downloaded from the course drive: 

[usr-topicalchat-roberta_ft.zip](https://drive.google.com/file/d/1ODF-trnYeWm_hSkv9-D6-xHlnM47ToBy/view?usp=share_link)

Unpack the zip file and place it anywhere on your local machine. Adapt the path in this notebook below to your local copy of the model:

You can use any other BERT or RoBERTa model (ENCODER) from [hugggingface.co](https://huggingface.co) to score the likelihood. 


Each system token of the uterance under consideration is turned into a masked token to predict the most probable tokens according to the model (cut-off by ```len_top_tokens```). If the system token is in the list, it will receive the score from the model. If it is not in the list, the token scores ```0```.

## Prerequisites

This notebooks relies on the transformers package and the EMISSOR package for loading the EMISSOR scenarios. These packages can be installed through ```pip```:

In [1]:
#!pip install emissor
#!pip install numpy==1.26.3
# !pip install torch==2.2.2
# !pip install transformers==4.51.3

## Loading an ENCODER model for the masked-task from huggingface or disk

In [2]:
import re
import numpy
from transformers import pipeline, AutoTokenizer

model_name = "google-bert/bert-base-uncased"
#model_name = "FacebookAI/xlm-roberta-base"
#model_name = "FacebookAI/roberta-base"

# Path to your local copy of the USR model
model_name ="../../models/usr-topicalchat-roberta_ft"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = pipeline("fill-mask", model=model_name)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/evaluate/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/evaluate/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/opt/anaconda3/envs/evaluate/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/evaluate/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
 

In [3]:
def mask_target_sentence(context, target):
    masked_targets = []
    ## We limit the length of the target as too long utterance break the token limit
    target_tokens = re.split(' ', target[:500])
    for index, token in enumerate(target_tokens):
        sequence = context + " "
        for token in target_tokens[:index]:
            sequence += token + " "
        sequence += tokenizer.mask_token
        for token in target_tokens[index + 1:]:
            sequence += " " + token
        masked_targets.append(sequence)
    return masked_targets, target_tokens

def sentence_likelihood(context, target):
    masked_targets, target_tokens = mask_target_sentence(context, target)
    expected_target = ""
    max_scores = []
    scores = []
    for masked_target, token in zip(masked_targets, target_tokens):
        results = model(masked_target)
        expected_target += results[0]['token_str'] + " "
        max_scores.append(results[0]['score'])
        match = False
        for result in results:
            if result['token_str'].lower().strip() == token.lower():
                scores.append(result['score'])
                match = True
                break
        if not match:
            scores.append(0)
    likelihood = sum(scores) / len(scores)
    max_likelihood = sum(max_scores) / len(max_scores)

    return likelihood, expected_target, max_likelihood

def score_pairs_for_likelihood(turns: []):
    for context, target in turns:
        llh, best_sentence, max_score = sentence_likelihood(context, target)
        print('Likelihood:', llh, 'Max score:', max_score, 'Best sentence:', best_sentence)

## Obtaining the conversations from EMISSOR

The next function gets the text signals from a conversation captured in an EMISSOR scenario.

In [4]:
import os
from emissor.persistence import ScenarioStorage
from emissor.representation.scenario import Modality
from emissor.representation.scenario import Signal, TextSignal
import emissor_util as util

In [5]:
EMISSOR="../emissor"
SCENARIO="b387db06-934e-405b-9d4e-f7e5c27b440a"

text_signals = util.get_text_signals_from_a_scenario(EMISSOR, SCENARIO)
llh_results = []
text = ""
context= ""
for index, text_signal in enumerate(text_signals):
    print(f"Processing turn {index}/{len(text_signals) - 1}")
    turn_id = text_signal.id
    speaker = util.get_speaker_from_text_signal(text_signal)
    if index==0:
        content = ""
    else:
        context = target
    text = text_signal.text
    llh, model_sentence, max_score = sentence_likelihood(context, text)
    row = {"Turn": turn_id, "Speaker": speaker, "Response": text, "llh": llh, "model_turn": model_sentence, "max_llh": max_score}
    llh_results.append(row)

df = pd.DataFrame(llh_results)

evaluation_folder = os.path.join(emissor, scenario_id, 'evaluation')
if not os.path.exists(evaluation_folder):
    os.mkdir(evaluation_folder)
file_name =  scenario_id+"_likelihood_evaluation.csv"
file = os.path.join(evaluation_folder, file_name)
df.to_csv(file, sep=";", index=False)

Processing turn 0/8


RuntimeError: Numpy is not available

In [18]:
speaker_llh = []
agent_llh = []
speaker_max_llh = []
agent_max_llh = []
for llh in llh_results:
    print(llh["turn"], ":", llh["model_turn"])
    if llh["speaker"]=="LEOLANI":
        agent_llh.append(llh["llh"])
        agent_max_llh.append(llh["max_llh"])
    else:
        speaker_llh.append(llh["llh"])
        speaker_max_llh.append(llh["max_llh"])

average_speaker_llh = sum(speaker_llh)/len(speaker_llh)
average_speaker_max_llh = sum(speaker_max_llh)/len(speaker_max_llh)
average_agent_llh = sum(agent_llh)/len(agent_llh)
average_agent_max_llh = sum(agent_max_llh)/len(agent_max_llh)

print('average_speaker_llh',average_speaker_llh)
print('average_speaker_max_llh',average_speaker_max_llh)
print('average_agent_llh',average_agent_llh)
print('average_agent_max_llh',average_agent_max_llh)

Lucy drinks wine :  and  of  ? 
Sorry, I did not get that. :  .  i  do  not  know  that 
Lucy is my friend : she  is  my  favorite 
I see. This is what I got from what you said:  (wilbur, get, that) (wilbur, get, wine) (wilbur, drink, wine) (wilbur, be, wilbur-'-s-friend)  :  you  think  this  is  what  i  got  from  what  you  said  wine  (  do  wine  (  drink  that  (  get  )  ,  '  that  . 
I know. I have heard about you before :  i  think i  have  heard  of  you  . 
I know Lucy :  you  like  . 
I see. Cannot make much of what you said. :  i  did  you  remember  sense  of  what  he  say 
Fred knows Lucy too : everyone  and  that  . 
Ok, so:  (wilbur, know, lucy) (wilbur, know, lucy) (wilbur, can-make-much-of, lucy-what-leolani-said) (wilbur, can-make-much-of, lucy)  :  .  " Fred  (  know    (  know    (  tell    (  know  and  ) 
Who knows Lucy? :  ,  is  ? 
Ok, so:  (, know, lucy) (, know, lucy)  :  or  Lucy  Lucy  ,  like  )  (  know  )  ? 
I am out of words. :  i  ran  afraid  of 

## End of notebook

In [ ]:

def make_overview_csv_any_scenario(emissor, scenario):
    overviewdf= pd.DataFrame()
    llh_files = glob.glob(folder + "/**/likelihood_evaluation*_overall.csv", recursive=True)
    scenario_dict = {}
    for f in llh_files:
        #file = open(f, 'r')
        evaluation_dir = os.path.dirname(f)
        scenario_dir = os.path.dirname(evaluation_dir)
        scenario_id = os.path.basename(scenario_dir)
        meta_file_path = os.path.join(evaluation_dir, scenario_id+"_meta_data.json")
        print(f)
        print(evaluation_dir)
        print(scenario_dir)
        print(meta_file_path)
        speaker = "None"
        model = "None"
        for m in models:
            if m in f:
                model = m
                print('MODEL', model)
                break
        nr_turns = 0
        meta_file = open(meta_file_path, 'r')
        meta_data = json.load(meta_file)
        if "Scenario" in meta_data:
            speaker = meta_data["Scenario"]["Speaker"] if "Speaker" in meta_data["Scenario"] else "None"
        if "Text" in meta_data:
            nr_turns = meta_data["Text"]["Nr. of signals"] if "Nr. of signals" in meta_data["Text"] else 0
        overviewrow = {"speaker": speaker, "scenario": scenario_id}
        df = pd.read_csv(f)
        for index, row in df.iterrows():
            if row['Speaker'].lower() == 'speaker':
                overviewrow.update({"Turns": row['Nr. turns']})
                column_name_LLH = "Speaker_"+model+"_"+"LLH"
                column_name_MAX = "Speaker_"+model+"_"+"MAX"

                overviewrow.update({column_name_LLH: row['MLM avg']})
                overviewrow.update({column_name_MAX: row['MLM avg max']})
            elif row['Speaker'].lower() == 'leolani':
                column_name_LLH = "Leolani_"+model+"_"+"LLH"
                column_name_MAX = "Leolani_"+model+"_"+"MAX"
                overviewrow.update({column_name_LLH: row['MLM avg']})
                overviewrow.update({column_name_MAX: row['MLM avg max']})
       # print(overviewrow)
        if scenario_id in scenario_dict:
            scenario_dict[scenario_id].append(overviewrow)
        else:
            scenario_dict[scenario_id]=[overviewrow]
    for scenario_id in scenario_dict.keys():
        #print(scenario_id)
        data = scenario_dict[scenario_id]
        row = {}
        for d in data:
            for v in d:
               # print('v', v)
                if not v in row:
                    row.update({v:d[v]})
        overviewdf = overviewdf._append(row, ignore_index=True)
